# Backup Lifetime Visualization — Concept Demo

This notebook is **not** wired into the Trier dataset or the NCNN pipeline — it's a small,
self-contained mock-up (synthetic data, plain SVG + JS, no Leaflet, no external calls). It now
implements the full 4-state mechanism from
[`plan/Logic.md`](plan/Logic.md) and the rendering rules from
[`plan/visual.md`](plan/visual.md), so this is the place to actually see the finished design
working end to end before any of it touches the real pipeline.

| Idea | Where to look |
|---|---|
| **Shape = facility type** | Circle = hospital, triangle = fire station |
| **4-state color** | Green `Operational`, amber `Depleting`, grey `Dead`, violet **blinking** `Rebooting` — no ring, no countdown on the map for reboot progress, per visual.md §13 |
| **Dual reserve rings** | Inner = power buffer, outer = water buffer — each grows/shrinks gradually (`gain_rate`/`loss_rate`), frozen while `Dead`/`Rebooting`, turns grey only once that specific buffer is at 0 |
| **Binary connection lines** | Brand color when that infrastructure node is connected, grey when it's down — independent of the building's FSM state |
| **Click-to-inspect panel** | Full detail per facility: state, both resources' connectivity/buffer/%, why a `Dead` building isn't rebooting yet, reboot progress while `Rebooting` |
| **Aggregate status bar** | Live counts for all four states across both facilities |

Press **Play**. Both facilities lose power, then water, deplete their buffers, go `Dead`,
recover enough to start `Rebooting` (blinking), and — because water is still down when the
reboot finishes — land back in `Depleting` rather than `Operational`, then fail a second time
before finally recovering for good. This two-cycle story falls out of the state machine itself
from a fairly simple scripted outage schedule; nothing about it was hand-authored per state.


## 1. Toy simulation data

No geodata, no OSM, no Trier — just a scripted outage schedule and the per-resource buffer
parameters from Logic.md §4 (`buffer_capacity`, `loss_rate`, `gain_rate`, `recharge_delay`,
`restart_threshold`), one set per facility type.

In [1]:
import json
from pathlib import Path
from IPython.display import HTML, display

HOURS = 48

# Hours during which each infrastructure node is itself down (e.g. flooded).
INFRA_DOWN_WINDOWS = {
    "power": (10, 26),
    "water": (16, 40),
}

# Per-facility-type parameters (Logic.md §4). `recharge_delay` and the shared
# `RESTART_THRESHOLD` are building-level; capacity/loss_rate/gain_rate are
# tracked separately per resource (Logic.md §1, "Dual-Resource Tracking").
BACKUP_CFG = {
    "hospital": {
        "recharge_delay": 5,
        "resources": {
            "power": {"capacity": 8, "loss_rate": 1, "gain_rate": 1},
            "water": {"capacity": 6, "loss_rate": 1, "gain_rate": 1},
        },
    },
    "fire_station": {
        "recharge_delay": 3,
        "resources": {
            "power": {"capacity": 4, "loss_rate": 1, "gain_rate": 1},
            "water": {"capacity": 2, "loss_rate": 1, "gain_rate": 1},
        },
    },
}
RESTART_THRESHOLD = 0.15  # Logic.md §4/§5 hysteresis guard

INFRA_META = {
    "power": {"label": "Power Station", "color": "#FF8C00"},
    "water": {"label": "Water Station", "color": "#00AEEF"},
}
POI_META = {
    "hospital":     {"label": "Hospital"},
    "fire_station": {"label": "Fire Station"},
}


def connected_by_hour(infra_type: str):
    start, end = INFRA_DOWN_WINDOWS[infra_type]
    return [not (start <= h < end) for h in range(HOURS)]


infra_connected = {t: connected_by_hour(t) for t in INFRA_META}
print({t: "".join("." if c else "#" for c in infra_connected[t]) for t in infra_connected})
print("(. = connected, # = down)")


{'power': '..........################......................', 'water': '................########################........'}
(. = connected, # = down)


## 2. Per-hour Finite State Machine

Direct implementation of Logic.md §3, §6 and §7: each facility carries its own power/water
buffers and a single shared `reboot_timer`; per hour, buffers update first (decay/refill, or
stay frozen while `Dead`/`Rebooting`), then the state transition is evaluated against the
*updated* buffers — exactly the Simulation Transition Matrix in Logic.md §7.

* **Standard Viability** (stay `Operational`/`Depleting`): for each resource,
  `connected OR buffer > 0`.
* **Restart Viability** (`Dead` → `Rebooting`): for each resource,
  `connected OR buffer > restart_threshold * capacity`.
* Buffers freeze the instant a building enters `Dead` or `Rebooting`, and only resume
  decaying/refilling once it's back to `Operational`/`Depleting`.

In [2]:
def simulate_poi(poi_type: str) -> dict:
    cfg = BACKUP_CFG[poi_type]
    resources = cfg["resources"]
    buffers = {r: resources[r]["capacity"] for r in resources}  # start full
    state = "Operational"
    reboot_timer = 0

    state_by_hour = []
    reboot_timer_by_hour = []
    buffer_by_hour = {r: [] for r in resources}

    for h in range(HOURS):
        connected = {r: infra_connected[r][h] for r in resources}

        if state in ("Dead", "Rebooting"):
            pass  # buffers frozen — Logic.md §3
        else:
            for r, rc in resources.items():
                if connected[r]:
                    buffers[r] = min(rc["capacity"], buffers[r] + rc["gain_rate"])
                else:
                    buffers[r] = max(0, buffers[r] - rc["loss_rate"])

        if state == "Operational":
            state = "Operational" if all(connected.values()) else "Depleting"
            reboot_timer = 0
        elif state == "Depleting":
            standard_ok = all(connected[r] or buffers[r] > 0 for r in resources)
            if not standard_ok:
                state = "Dead"
            elif all(connected.values()):
                state = "Operational"
            else:
                state = "Depleting"
            reboot_timer = 0
        elif state == "Dead":
            restart_ok = all(
                connected[r] or buffers[r] > RESTART_THRESHOLD * resources[r]["capacity"]
                for r in resources
            )
            if restart_ok:
                state = "Rebooting"
                reboot_timer = 1
            else:
                reboot_timer = 0
        elif state == "Rebooting":
            restart_ok = all(
                connected[r] or buffers[r] > RESTART_THRESHOLD * resources[r]["capacity"]
                for r in resources
            )
            if not restart_ok:
                state = "Dead"
                reboot_timer = 0
            else:
                reboot_timer += 1
                if reboot_timer >= cfg["recharge_delay"]:
                    state = "Operational" if all(connected.values()) else "Depleting"
                    reboot_timer = 0

        state_by_hour.append(state)
        reboot_timer_by_hour.append(reboot_timer)
        for r in resources:
            buffer_by_hour[r].append(buffers[r])

    return {
        "label": POI_META[poi_type]["label"],
        "recharge_delay": cfg["recharge_delay"],
        "state_by_hour": state_by_hour,
        "reboot_timer_by_hour": reboot_timer_by_hour,
        "deps": {
            r: {
                "capacity": resources[r]["capacity"],
                "buffer_by_hour": buffer_by_hour[r],
                "fraction_by_hour": [b / resources[r]["capacity"] for b in buffer_by_hour[r]],
            }
            for r in resources
        },
    }


pois = {poi_type: simulate_poi(poi_type) for poi_type in POI_META}
infra = {
    infra_type: {"label": meta["label"], "color": meta["color"], "connected_by_hour": infra_connected[infra_type]}
    for infra_type, meta in INFRA_META.items()
}

_CODE = {"Operational": "O", "Depleting": "D", "Rebooting": "R", "Dead": "X"}
for poi_type, poi in pois.items():
    print(f"{poi_type:14s} {''.join(_CODE[s] for s in poi['state_by_hour'])}")
print("(O=Operational D=Depleting R=Rebooting X=Dead)")


hospital       OOOOOOOOOODDDDDDDXXXXXXXXXRRRRDDDDXXXXXXRRRROOOO
fire_station   OOOOOOOOOODDDXXXXXXXXXXXXXRRDDXXXXXXXXXXRROOOOOO
(O=Operational D=Depleting R=Rebooting X=Dead)


## 3. Rendering

Plain inline SVG + vanilla JS (no Leaflet, no CDN) so this renders anywhere without network
access. In the real pipeline this would sit inside the existing Leaflet-based
`build_flood_animation_html` (`utils/flood_interpolation.py`) using the same technique — an
`L.divIcon` with this SVG for markers, `L.polyline` with dynamic `setStyle()` for the
connection lines — driven by per-hour arrays computed the same way as `pois`/`infra` above.

**Type vs. status.** Marker color is spoken for by state (green/amber/grey/violet), so
facility *type* can't also be a color the way `system_overview.ipynb` does today (red
hospitals, blue fire stations) without the two colliding. This demo uses **shape** instead
(circle = hospital, triangle = fire station) — it scales to more facility types later
(square, diamond, ...) and stays legible at small map zoom.

In [3]:
_FRAGMENT_TEMPLATE = r"""
<div class="backup-demo-wrap">
  <style>
    .backup-demo-wrap { font-family: -apple-system, "Segoe UI", Arial, sans-serif; background:#0f1115; color:#e8e8ee; padding:16px; border-radius:10px; max-width:760px; }
    .backup-demo-stage { position:relative; }
    .backup-demo-wrap svg.demo-svg { width:100%; height:auto; background:#161a20; border-radius:8px; display:block; }
    .backup-demo-wrap .controls { display:flex; align-items:center; gap:10px; margin-top:12px; flex-wrap:wrap; }
    .backup-demo-wrap button.demo-btn { background:#2a2f3a; color:#fff; border:1px solid #3a4150; border-radius:6px; padding:6px 12px; cursor:pointer; font-size:13px; }
    .backup-demo-wrap button.demo-btn:hover { background:#3a4150; }
    .backup-demo-wrap input[type=range] { flex:1; min-width:160px; }
    .backup-demo-wrap .hour-readout { font-variant-numeric: tabular-nums; min-width:78px; text-align:right; font-size:13px; }
    .backup-demo-wrap .status-bar { display:flex; gap:16px; flex-wrap:wrap; margin-top:10px; font-size:12px; color:#c7ccd8; }
    .backup-demo-wrap .status-bar b { color:#fff; }
    .backup-demo-wrap .legend { display:flex; gap:16px; flex-wrap:wrap; margin-top:10px; font-size:12px; color:#aab0c0; }
    .backup-demo-wrap .legend .dot { display:inline-block; width:10px; height:10px; border-radius:50%; margin-right:5px; vertical-align:middle; }
    .backup-demo-wrap .panel { position:absolute; top:10px; right:10px; width:230px; background:rgba(20,23,30,0.95); border:1px solid #3a4150; border-radius:8px; padding:10px 12px; font-size:12px; }
    .backup-demo-wrap .panel h4 { margin:0 0 8px 0; font-size:13px; }
    .backup-demo-wrap .panel .row { display:flex; justify-content:space-between; margin:4px 0; gap:8px; }
    .backup-demo-wrap .panel .hint { color:#8890a5; font-style:italic; margin-top:6px; }
    .backup-demo-wrap .poi-marker { cursor:pointer; }
    @keyframes demo-blink { 0%, 49% { opacity: 1; } 50%, 100% { opacity: 0.15; } }
    .backup-demo-wrap .blinking { animation: demo-blink 1s steps(1) infinite; }
  </style>

  <div class="backup-demo-stage">
    <svg class="demo-svg" viewBox="0 0 640 380" xmlns="http://www.w3.org/2000/svg">
      <line id="conn-hospital-power"     x1="150" y1="70" x2="150" y2="282" stroke="#FF8C00" stroke-width="3"/>
      <line id="conn-hospital-water"     x1="490" y1="70" x2="168" y2="292" stroke="#00AEEF" stroke-width="3"/>
      <line id="conn-fire_station-power" x1="150" y1="70" x2="472" y2="292" stroke="#FF8C00" stroke-width="3"/>
      <line id="conn-fire_station-water" x1="490" y1="70" x2="490" y2="282" stroke="#00AEEF" stroke-width="3"/>

      <rect id="infra-power" x="134" y="54" width="32" height="32" rx="6" fill="#FF8C00" stroke="#fff" stroke-width="2"/>
      <text x="150" y="106" text-anchor="middle" fill="#ccd0dc" font-size="13">Power</text>
      <rect id="infra-water" x="474" y="54" width="32" height="32" rx="6" fill="#00AEEF" stroke="#fff" stroke-width="2"/>
      <text x="490" y="106" text-anchor="middle" fill="#ccd0dc" font-size="13">Water</text>

      <circle cx="150" cy="300" r="32" fill="none" stroke="#444a58" stroke-width="4" opacity="0.35"/>
      <circle cx="150" cy="300" r="25" fill="none" stroke="#444a58" stroke-width="4" opacity="0.35"/>
      <circle id="ring-hospital-water" cx="150" cy="300" r="32" fill="none" stroke="#00AEEF" stroke-width="4" stroke-linecap="round" transform="rotate(-90 150 300)"/>
      <circle id="ring-hospital-power" cx="150" cy="300" r="25" fill="none" stroke="#FF8C00" stroke-width="4" stroke-linecap="round" transform="rotate(-90 150 300)"/>
      <circle id="marker-hospital" class="poi-marker" data-poi="hospital" cx="150" cy="300" r="18" fill="#2ecc71" stroke="#fff" stroke-width="2"/>
      <text x="150" y="346" text-anchor="middle" fill="#ccd0dc" font-size="13">Hospital</text>

      <circle cx="490" cy="300" r="32" fill="none" stroke="#444a58" stroke-width="4" opacity="0.35"/>
      <circle cx="490" cy="300" r="25" fill="none" stroke="#444a58" stroke-width="4" opacity="0.35"/>
      <circle id="ring-fire_station-water" cx="490" cy="300" r="32" fill="none" stroke="#00AEEF" stroke-width="4" stroke-linecap="round" transform="rotate(-90 490 300)"/>
      <circle id="ring-fire_station-power" cx="490" cy="300" r="25" fill="none" stroke="#FF8C00" stroke-width="4" stroke-linecap="round" transform="rotate(-90 490 300)"/>
      <polygon id="marker-fire_station" class="poi-marker" data-poi="fire_station" points="490,279 471.8,310.5 508.2,310.5" fill="#2ecc71" stroke="#fff" stroke-width="2" stroke-linejoin="round"/>
      <text x="490" y="346" text-anchor="middle" fill="#ccd0dc" font-size="13">Fire Station</text>
    </svg>

    <div class="panel" id="demo-panel">
      <h4>Click a facility</h4>
      <div class="hint">Select a marker to see its live connection &amp; backup status.</div>
    </div>
  </div>

  <div class="controls">
    <button class="demo-btn" id="demo-play-btn">&#9654; Play</button>
    <button class="demo-btn" id="demo-reset-btn">&#8634; Reset</button>
    <input type="range" id="demo-hour-slider" min="0" max="__MAX_HOUR__" value="0" step="1"/>
    <span class="hour-readout" id="demo-hour-readout">Hour 0 / __MAX_HOUR__</span>
  </div>

  <div class="status-bar" id="demo-status-bar"></div>

  <div class="legend">
    <span>&#9679; Hospital &nbsp; &#9650; Fire Station <em>(shape = type)</em></span>
    <span><span class="dot" style="background:#2ecc71"></span>Operational</span>
    <span><span class="dot" style="background:#f39c12"></span>Depleting</span>
    <span><span class="dot" style="background:#8a5cf6"></span>Rebooting (blinking)</span>
    <span><span class="dot" style="background:#8a8a8a"></span>Dead</span>
    <span>Inner ring = power reserve &middot; outer ring = water reserve</span>
    <span>Grey line = that infrastructure link is down</span>
  </div>

  <script>
  (function () {
    var HOURS = __HOURS__;
    var RESTART_THRESHOLD = __RESTART_THRESHOLD__;
    var INFRA = __INFRA_JSON__;
    var POIS = __POIS_JSON__;
    var STATE_COLOR = { Operational: '#2ecc71', Depleting: '#f39c12', Dead: '#8a8a8a', Rebooting: '#8a5cf6' };
    var STATE_ORDER = ['Operational', 'Depleting', 'Rebooting', 'Dead'];
    var RING_R = { power: 25, water: 32 };
    var RING_CIRC = {};
    Object.keys(RING_R).forEach(function (k) { RING_CIRC[k] = 2 * Math.PI * RING_R[k]; });

    var hour = 0, playing = false, timer = null, selected = null;
    var slider = document.getElementById('demo-hour-slider');
    var readout = document.getElementById('demo-hour-readout');
    var playBtn = document.getElementById('demo-play-btn');
    var panelEl = document.getElementById('demo-panel');
    var statusBarEl = document.getElementById('demo-status-bar');

    function pct(x) { return Math.round(x * 100) + '%'; }

    function updatePanel(poiType, h) {
      var poi = POIS[poiType];
      var state = poi.state_by_hour[h];
      var html = '<h4>' + poi.label + '</h4>';
      html += '<div class="row"><span>State</span><b style="color:' + STATE_COLOR[state] + '">' + state + '</b></div>';

      var restartNotes = [];
      Object.keys(poi.deps).forEach(function (infraType) {
        var dep = poi.deps[infraType];
        var buffer = dep.buffer_by_hour[h];
        var capacity = dep.capacity;
        var frac = capacity ? buffer / capacity : 0;
        var connected = INFRA[infraType].connected_by_hour[h];

        var dynLabel;
        if (state === 'Dead' || state === 'Rebooting') dynLabel = 'frozen';
        else if (connected) dynLabel = (buffer < capacity ? 'refilling' : 'full');
        else dynLabel = 'decaying';

        html += '<div class="row"><span>' + INFRA[infraType].label + '</span><span>' + (connected ? 'connected' : 'down') + '</span></div>';
        html += '<div class="row"><span>Buffer</span><span>' + buffer + ' / ' + capacity + ' (' + pct(frac) + ', ' + dynLabel + ')</span></div>';

        var restartOk = connected || frac > RESTART_THRESHOLD;
        if (!restartOk) {
          restartNotes.push(INFRA[infraType].label + ' ' + pct(frac) + ' &le; ' + pct(RESTART_THRESHOLD) + ', still disconnected');
        }
      });

      if (state === 'Dead') {
        html += '<div class="hint">Restart blocked &mdash; ' + restartNotes.join('; ') + '.</div>';
      }
      if (state === 'Rebooting') {
        var rt = poi.reboot_timer_by_hour[h];
        html += '<div class="row"><span>Reboot progress</span><span>' + rt + ' / ' + poi.recharge_delay + ' h</span></div>';
        html += '<div class="hint">~' + (poi.recharge_delay - rt) + ' h until back online.</div>';
      }
      html += '<div class="hint">Click another facility to switch selection.</div>';
      panelEl.innerHTML = html;
    }

    function updateStatusBar(h) {
      var counts = { Operational: 0, Depleting: 0, Rebooting: 0, Dead: 0 };
      Object.keys(POIS).forEach(function (poiType) { counts[POIS[poiType].state_by_hour[h]]++; });
      statusBarEl.innerHTML = STATE_ORDER.map(function (s) {
        return '<span><span class="dot" style="background:' + STATE_COLOR[s] + '"></span>' + s + ': <b>' + counts[s] + '</b></span>';
      }).join('');
    }

    function render(h) {
      Object.keys(INFRA).forEach(function (infraType) {
        var connected = INFRA[infraType].connected_by_hour[h];
        var el = document.getElementById('infra-' + infraType);
        el.setAttribute('fill', connected ? INFRA[infraType].color : '#5a5a5a');
        el.setAttribute('opacity', connected ? '1' : '0.55');
      });

      Object.keys(POIS).forEach(function (poiType) {
        var poi = POIS[poiType];
        var state = poi.state_by_hour[h];
        var marker = document.getElementById('marker-' + poiType);
        marker.setAttribute('fill', STATE_COLOR[state]);
        marker.classList.toggle('blinking', state === 'Rebooting');

        Object.keys(poi.deps).forEach(function (infraType) {
          var dep = poi.deps[infraType];
          var frac = dep.fraction_by_hour[h];
          var ring = document.getElementById('ring-' + poiType + '-' + infraType);
          var circ = RING_CIRC[infraType];
          ring.setAttribute('stroke-dasharray', circ);
          ring.setAttribute('stroke-dashoffset', circ * (1 - frac));
          ring.setAttribute('stroke', frac <= 0 ? '#5a5a5a' : INFRA[infraType].color);

          var connected = INFRA[infraType].connected_by_hour[h];
          var line = document.getElementById('conn-' + poiType + '-' + infraType);
          line.setAttribute('stroke', connected ? INFRA[infraType].color : '#5a5a5a');
          line.setAttribute('stroke-width', connected ? 3 : 2);
          line.setAttribute('opacity', connected ? 0.9 : 0.45);
        });
      });

      readout.textContent = 'Hour ' + h + ' / ' + (HOURS - 1);
      slider.value = h;
      updateStatusBar(h);
      if (selected) updatePanel(selected, h);
    }

    document.querySelectorAll('.poi-marker').forEach(function (el) {
      el.addEventListener('click', function () {
        selected = el.getAttribute('data-poi');
        updatePanel(selected, hour);
      });
    });

    slider.addEventListener('input', function () {
      hour = parseInt(slider.value, 10);
      render(hour);
    });

    playBtn.addEventListener('click', function () {
      playing = !playing;
      playBtn.innerHTML = playing ? '&#10074;&#10074; Pause' : '&#9654; Play';
      if (playing) {
        timer = setInterval(function () {
          hour = (hour + 1) % HOURS;
          render(hour);
        }, 350);
      } else {
        clearInterval(timer);
      }
    });

    document.getElementById('demo-reset-btn').addEventListener('click', function () {
      playing = false; clearInterval(timer); playBtn.innerHTML = '&#9654; Play';
      hour = 0; render(hour);
    });

    render(hour);
  })();
  </script>
</div>
"""


def build_fragment() -> str:
    html = _FRAGMENT_TEMPLATE
    html = html.replace("__HOURS__", json.dumps(HOURS))
    html = html.replace("__MAX_HOUR__", str(HOURS - 1))
    html = html.replace("__RESTART_THRESHOLD__", json.dumps(RESTART_THRESHOLD))
    html = html.replace("__INFRA_JSON__", json.dumps(infra, separators=(",", ":")))
    html = html.replace("__POIS_JSON__", json.dumps(pois, separators=(",", ":")))
    return html


display(HTML(build_fragment()))


## 4. Save a standalone copy

Same fragment, wrapped in a full HTML document, so it can be opened directly in a browser or shared without Jupyter.

In [4]:
standalone_html = (
    "<!DOCTYPE html><html><head><meta charset='utf-8'>"
    "<title>Backup Lifetime Visualization Demo</title></head>"
    "<body style='background:#0f1115;margin:0;padding:24px;'>" + build_fragment() + "</body></html>"
)

out_path = Path("backup_lifetime_demo.html")
out_path.write_text(standalone_html, encoding="utf-8")
print(f"Saved: {out_path.resolve()}")


Saved: C:\Users\WelJo\IdeaProjects\forschungspraktikum\code\css_geodata_service\robustness_of_accessibility\examples\backup_lifetime_demo.html


---

Naming and behaviour here now match [`plan/Logic.md`](plan/Logic.md) (the simulation engine)
and [`plan/visual.md`](plan/visual.md) (this rendering spec) exactly — `Operational` /
`Depleting` / `Dead` / `Rebooting`, dual per-resource buffers with independent
`gain_rate`/`loss_rate`, the 15% hysteresis guard, and a blink-only (no ring) `Rebooting`
indicator with full detail on click. This replaces the earlier `up`/`backup`/`dead`
placeholder model from the first iteration of this notebook.